In [1]:
import os
import yaml
import pytz
import requests
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# import functions from bookie_grabber
from bookie_grabber import *

load_dotenv()

True

In [ ]:
events = get_league_events(api_key)
df_events = extract_events_to_df(events)


perth_tz = pytz.timezone("Australia/Perth")
now = datetime.now(perth_tz)

# Only include matches within ±30 minutes of exactly 9 hours from KO
df_events = df_events[
    (df_events["match_time"] - now).dt.total_seconds().between(9*3600 - 1800, 9*3600 + 1800)
].reset_index(drop=True)

print(f"Filtered to {len(df_events)} matches ~9 hours before KO.")

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  1. SPORTS                                              │
# │  What sports are available?                             │
# │  GET /v3/sports                                         │
# │  Returns: Football, Basketball, Tennis, etc.            │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  2. LEAGUES                                             │
# │  What leagues exist in a sport?                         │
# │  GET /v3/leagues?sport=football                         │
# │  Returns: Premier League, La Liga, Bundesliga, etc.     │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  3. EVENTS                                              │
# │  What matches are coming up?                            │
# │  GET /v3/events?sport=football&league=premier-league    │
# │  Returns: Man Utd vs Liverpool, Chelsea vs Arsenal, etc.│
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  4. ODDS                                                │
# │  What are the betting odds?                             │
# │  GET /v3/odds?eventId=123456&bookmakers=Bet365,Unibet   │
# │  Returns: Odds from multiple bookmakers                 │
# └─────────────────────────────────────────────────────────┘

In [2]:
# "status": "pending" for events
# Example usage
df_totals, df_btts = main()

Fetching odds for Fulham FC vs Manchester City
Fetching odds for AFC Bournemouth vs Everton FC
Fetching odds for Newcastle United vs Tottenham Hotspur


KeyboardInterrupt: 

In [ ]:
df = pd.read_csv("/Users/notbahd/Desktop/BookieGrabber/data/exports/totals/totals_hdp_2.5_20251123.csv")
df

In [ ]:
# Hey mate, the volume is already available, listed under “depth” in the markets array.

In [ ]:
odds = get_event_odds(api_key, 61300735)

In [ ]:
odds.keys()

In [ ]:
odds.get("markets", {})

In [ ]:
# def extract_totals(odds_data):
#     """Extract Totals market from odds data."""
#     totals = []
for bookmaker, markets in odds.get("bookmakers", {}).items():
    allowed_markets = TOTAL_MARKETS.get(bookmaker, [])
    for market in markets:
        # Skip irrelevant markets
        if market.get("name") not in allowed_markets:
            continue
        for o in market.get("odds", []):
            hdp = o.get("hdp")
            if hdp not in TARGET_HDPS:
                continue
            totals.append({
                "bookmaker": bookmaker,
                "market_name": market.get("name"),
                "hdp": hdp,
                "over_odds": o.get("over"),
                "under_odds": o.get("under")
            })
    # return totals